In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/mvsamultiple/MVSA/labelResultAll.txt
/kaggle/input/mvsamultiple/MVSA/data/22341.txt
/kaggle/input/mvsamultiple/MVSA/data/20430.txt
/kaggle/input/mvsamultiple/MVSA/data/19812.jpg
/kaggle/input/mvsamultiple/MVSA/data/22735.jpg
/kaggle/input/mvsamultiple/MVSA/data/16916.jpg
/kaggle/input/mvsamultiple/MVSA/data/7360.txt
/kaggle/input/mvsamultiple/MVSA/data/7981.jpg
/kaggle/input/mvsamultiple/MVSA/data/13052.txt
/kaggle/input/mvsamultiple/MVSA/data/4682.txt
/kaggle/input/mvsamultiple/MVSA/data/5450.txt
/kaggle/input/mvsamultiple/MVSA/data/19963.txt
/kaggle/input/mvsamultiple/MVSA/data/11184.txt
/kaggle/input/mvsamultiple/MVSA/data/22706.jpg
/kaggle/input/mvsamultiple/MVSA/data/11776.txt
/kaggle/input/mvsamultiple/MVSA/data/20513.jpg
/kaggle/input/mvsamultiple/MVSA/data/5064.txt
/kaggle/input/mvsamultiple/MVSA/data/9932.txt
/kaggle/input/mvsamultiple/MVSA/data/12666.jpg
/kaggle/input/mvsamultiple/MVSA/data/17383.txt
/kaggle/input/mvsamultiple/MVSA/data/13288.jpg
/kaggle/input/m

# MVSA - Single

In [ ]:
import os
from PIL import Image
from collections import defaultdict, Counter

import pytesseract

import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

In [69]:
# Paths
image_dir = "/kaggle/input/mvsasingle/MVSA_Single/data"
label_file = "/kaggle/input/mvsasingle/MVSA_Single/labelResultAll.txt"

In [15]:
import re

sentiments = []
with open(label_file, "r", encoding="utf-8") as f:
    count = 0
    for line in f:
        if "ID" in line:
            continue
        count+=1
        parts = re.split(r'[\n,]', line)
        img_sentiment = parts[1]
        # print(img_sentiment)
        sentiments+=[img_sentiment]
        # break

In [16]:
print(len(sentiments))
print(count)

4869
4869


In [18]:
import os
# print(dir(os))

# print(os.listdir("/kaggle/input/mvsasingle/MVSA_Single/data"))

['1893.txt', '1711.txt', '4682.txt', '5064.txt', '3504.txt', '1269.jpg', '3863.jpg', '1773.txt', '623.jpg', '559.txt', '3750.jpg', '2008.jpg', '1812.txt', '1093.txt', '2081.jpg', '4417.txt', '3919.jpg', '4503.txt', '1356.txt', '2869.txt', '3926.txt', '2093.txt', '1880.txt', '3721.txt', '557.txt', '4489.jpg', '361.txt', '3138.jpg', '2842.txt', '1366.txt', '3417.jpg', '1866.txt', '5020.txt', '764.jpg', '5039.jpg', '4407.jpg', '1700.jpg', '1786.jpg', '1437.txt', '2907.jpg', '1075.jpg', '4969.jpg', '992.txt', '3079.txt', '3501.jpg', '2863.jpg', '771.jpg', '208.jpg', '3851.txt', '5120.txt', '2658.txt', '4640.jpg', '2975.txt', '4125.jpg', '2628.jpg', '3109.txt', '1038.txt', '40.txt', '3363.jpg', '1721.txt', '4009.jpg', '812.txt', '1025.txt', '245.txt', '2588.txt', '820.jpg', '3228.jpg', '870.txt', '1789.jpg', '655.txt', '5052.jpg', '3749.txt', '473.jpg', '3446.jpg', '4334.txt', '1792.jpg', '4772.jpg', '431.txt', '2936.jpg', '4424.txt', '2029.jpg', '4439.txt', '2460.txt', '4313.jpg', '3975.tx

In [55]:
# Define standard aspect ratios
standard_ratios = {
    "1:1": 1.0, # 1 BOX
    "4:3": 4/3, # 1.33 Horizontal
    "3:2": 3/2, # 1.5 Horizontal
    "16:9": 16/9, # 1.78 Horizontal
    "5:4": 5/4, # 1.25 Horizontal
    "21:9": 21/9, # 2.33 Horizontal
    "2:1": 2.0, # 2 Horizontal
    "7:5": 7/5, # 1.4 Horizontal
    # "A4 (1.41:1)": 1.41,
    "3:4": 3/4, # 0.75 Vertical
    "2:3": 2/3, # 0.67 Vertical
    "9:16": 9/16, # 0.5625 Vertical
    "5:7": 5/7, # 0.71 Vertical
    "4:5": 4/5, # 0.8 Vertical
    "A-series (1:√2)": 1/1.414, # 0.707 Vertical
    "2:5": 2/5, # 0.4 Vertical
    "Other":0
}

# Function to match image to closest standard aspect ratio
def closest_standard_ratio(width, height, tolerance=0.05):
    if height == 0:
        return "Invalid"
    ratio = width / height
    for name, std_ratio in standard_ratios.items():
        if abs(ratio - std_ratio) <= tolerance:
            return name
    return "Other"


ar_columnNames = list(standard_ratios.keys())
rowNames = ["positive", "negative", "neutral"]

ar_df = pd.DataFrame(0, index=rowNames, columns=ar_columnNames)

In [56]:
ar_df["4:3"]["positive"]

0

In [57]:
# list(standard_ratios.keys())

In [58]:
aspect_ratio_orientation = {
    # --- Box-like (almost square) ---
    "1:1": "box",

    # --- Horizontal (landscape) ---
    "4:3": "horizontal",      # 1.33
    "3:2": "horizontal",      # 1.5
    "16:9": "horizontal",     # 1.78
    "5:4": "horizontal",      # 1.25
    "21:9": "horizontal",     # 2.33
    "2:1": "horizontal",      # 2.0
    "7:5": "horizontal",      # 1.4
    "A4 (1.41:1)": "horizontal",  # 1.41

    # --- Vertical (portrait) ---
    "3:4": "vertical",        # 0.75
    "2:3": "vertical",        # 0.666
    "9:16": "vertical",       # 0.5625
    "5:7": "vertical",        # 0.714
    "4:5": "vertical",        # 0.8
    "A-series (1:√2)": "vertical",  # ≈0.707
    "2:5": "vertical",        # 0.4

    # --- Catch-all ---
    # "Other": "other"
    "Other": "box"
}



# Helper function to classify orientation
def get_orientation(aspect_ratio="Other"):
    return aspect_ratio_orientation[aspect_ratio]




columnNames = ["vertical", "horizontal","box"]
rowNames = ["positive", "negative", "neutral"]

# Create a DataFrame filled with zeros
orientation_df = pd.DataFrame(0, index=rowNames, columns=columnNames)

In [60]:
import os
from PIL import Image
from collections import defaultdict
import pandas as pd


files = os.listdir(image_dir)
files.sort()
files = [file for file in files if ".jpg" in file]

count = 0

for file in files:
    # fileno = file.split(".")[0]
    
    # print(file)
    
    filepath = os.path.join(image_dir, file)
    # print(filepath)
    # break
    img = Image.open(filepath)
    width, height = img.size
    ar = closest_standard_ratio(width, height)
    orientation = aspect_ratio_orientation[ar]
    ar_df.loc[sentiments[count],ar]+=1
    orientation_df.loc[sentiments[count],orientation]+=1
    count+=1
    # print(ar_df)
    # print(orientation_df)
    # break



print("DONE!")

DONE!


In [61]:
ar_df

,1:1,4:3,3:2,16:9,5:4,21:9,2:1,7:5,3:4,2:3,9:16,5:7,4:5,A-series (1:√2),2:5,Other
positive,594,359,228,227,46,5,62,60,382,138,167,0,44,0,5,391
negative,257,179,113,108,16,4,21,20,176,78,71,0,15,0,2,163
neutral,188,110,101,84,19,4,20,25,138,53,62,0,10,0,0,124


In [62]:
orientation_df

,vertical,horizontal,box
positive,736,987,985
negative,342,461,420
neutral,263,363,312


## Hypothesis Testing

In [64]:
import numpy as np
import pandas as pd

# Contingency table
# data = np.array([
#     [370, 242, 409],  # Positive
#     [53, 645, 209],   # Negative
#     [139, 419, 389]   # Neutral
# ])

data_mvsas = orientation_df.copy()

# Optional: Set labels
sentiments = ['positive', 'negative', 'neutral']
orientations = ['vertical', 'horizontal', 'box']

# df = pd.DataFrame(data, index=sentiments, columns=orientations)
# Contingency table

# Chi-square test
chi2, p, dof, expected = chi2_contingency(data_mvsas)

print(f"Chi-square statistic = {chi2:.3f}")
print(f"p-value = {p}")
print(f"Degrees of freedom = {dof}")

Chi-square statistic = 3.659
p-value = 0.45417407918006003
Degrees of freedom = 4


In [65]:
print("Expected frequencies:")
pd.DataFrame(np.round(expected, 2), index=sentiments, columns=orientations)

Expected frequencies:


,vertical,horizontal,box
positive,745.83,1007.23,954.95
negative,336.83,454.89,431.28
neutral,258.34,348.88,330.78


In [67]:
# p-value
p

0.45417407918006003

In [68]:
# Decision
alpha = 0.05
if p < alpha:
    print("Reject the null hypothesis: There is a relationship between orientation and sentiment.")
else:
    print("Fail to reject the null hypothesis: No significant relationship found.")

Fail to reject the null hypothesis: No significant relationship found.


# MVSA Multiple

In [70]:
# Paths
image_dir = "/kaggle/input/mvsamultiple/MVSA/data"
label_file = "/kaggle/input/mvsamultiple/MVSA/labelResultAll.txt"

In [82]:
import re
from collections import Counter

label_file = "/kaggle/input/mvsamultiple/MVSA/labelResultAll.txt"  # adjust path as needed

def get_majority_text(text_list):
    counts = Counter(text_list).most_common()
    if len(counts) > 1 and counts[0][1] == counts[1][1]:
        return "neutral"  # Tie case
    return counts[0][0]

sentiments = []
count = 0

with open(label_file, "r", encoding="utf-8") as f:
    for line in f:
        if "ID" in line:
            continue  # skip header
        count+=1
        parts = re.split(r'[\n\t]', line.strip())
        parts = list(filter(None, parts))  # remove empty strings
        annotators = parts[1:]  # ignore the ID
        img_sent = [ann.split(",")[1] for ann in annotators]  # get image sentiment
        majority_sentiment = get_majority_text(img_sent)
        sentiments.append(majority_sentiment)

# Example: print first few results
print(sentiments[:5])

['positive', 'positive', 'neutral', 'positive', 'neutral']


In [83]:
print(len(sentiments))
print(count)

19600
19600


In [84]:
import os
# print(dir(os))

# print(os.listdir("/kaggle/input/mvsasingle/MVSA_Single/data"))

In [90]:
# Define standard aspect ratios
standard_ratios = {
    "1:1": 1.0, # 1 BOX
    "4:3": 4/3, # 1.33 Horizontal
    "3:2": 3/2, # 1.5 Horizontal
    "16:9": 16/9, # 1.78 Horizontal
    "5:4": 5/4, # 1.25 Horizontal
    "21:9": 21/9, # 2.33 Horizontal
    "2:1": 2.0, # 2 Horizontal
    "7:5": 7/5, # 1.4 Horizontal
    # "A4 (1.41:1)": 1.41,
    "3:4": 3/4, # 0.75 Vertical
    "2:3": 2/3, # 0.67 Vertical
    "9:16": 9/16, # 0.5625 Vertical
    "5:7": 5/7, # 0.71 Vertical
    "4:5": 4/5, # 0.8 Vertical
    "A-series (1:√2)": 1/1.414, # 0.707 Vertical
    "2:5": 2/5, # 0.4 Vertical
    "Other":0
}

# Function to match image to closest standard aspect ratio
def closest_standard_ratio(width, height, tolerance=0.05):
    if height == 0:
        return "Invalid"
    ratio = width / height
    for name, std_ratio in standard_ratios.items():
        if abs(ratio - std_ratio) <= tolerance:
            return name
    return "Other"


ar_columnNames = list(standard_ratios.keys())
rowNames = ["positive", "negative", "neutral"]

ar_df = pd.DataFrame(0, index=rowNames, columns=ar_columnNames)

In [91]:
ar_df["4:3"]["positive"]

0

In [92]:
# list(standard_ratios.keys())

In [93]:
aspect_ratio_orientation = {
    # --- Box-like (almost square) ---
    "1:1": "box",

    # --- Horizontal (landscape) ---
    "4:3": "horizontal",      # 1.33
    "3:2": "horizontal",      # 1.5
    "16:9": "horizontal",     # 1.78
    "5:4": "horizontal",      # 1.25
    "21:9": "horizontal",     # 2.33
    "2:1": "horizontal",      # 2.0
    "7:5": "horizontal",      # 1.4
    "A4 (1.41:1)": "horizontal",  # 1.41

    # --- Vertical (portrait) ---
    "3:4": "vertical",        # 0.75
    "2:3": "vertical",        # 0.666
    "9:16": "vertical",       # 0.5625
    "5:7": "vertical",        # 0.714
    "4:5": "vertical",        # 0.8
    "A-series (1:√2)": "vertical",  # ≈0.707
    "2:5": "vertical",        # 0.4

    # --- Catch-all ---
    # "Other": "other"
    "Other": "box"
}



# Helper function to classify orientation
def get_orientation(aspect_ratio="Other"):
    return aspect_ratio_orientation[aspect_ratio]




columnNames = ["vertical", "horizontal","box"]
rowNames = ["positive", "negative", "neutral"]

# Create a DataFrame filled with zeros
orientation_df = pd.DataFrame(0, index=rowNames, columns=columnNames)

In [94]:
import os
from PIL import Image
from collections import defaultdict
import pandas as pd


files = os.listdir(image_dir)
files.sort()
files = [file for file in files if ".jpg" in file]

count = 0

for file in files:
    # fileno = file.split(".")[0]
    
    # print(file)
    
    filepath = os.path.join(image_dir, file)
    # print(filepath)
    # break
    try:
        img = Image.open(filepath)
        width, height = img.size
        ar = closest_standard_ratio(width, height)
        orientation = aspect_ratio_orientation[ar]
        ar_df.loc[sentiments[count],ar]+=1
        orientation_df.loc[sentiments[count],orientation]+=1
        count+=1
        # print(ar_df)
        # print(orientation_df)
        # break
    except:
        print(f"Skipping corrupted file: {file}")



print("DONE!")

Skipping corrupted file: 3151.jpg
Skipping corrupted file: 3910.jpg
Skipping corrupted file: 5995.jpg
DONE!


In [95]:
ar_df

,1:1,4:3,3:2,16:9,5:4,21:9,2:1,7:5,3:4,2:3,9:16,5:7,4:5,A-series (1:√2),2:5,Other
positive,1914,1623,600,670,173,35,624,127,1642,315,809,0,107,0,16,1276
negative,302,233,88,124,17,7,68,15,265,47,107,0,17,0,0,178
neutral,1626,1339,531,597,139,30,406,118,1375,295,624,0,84,0,14,1020


In [96]:
orientation_df

,vertical,horizontal,box
positive,2889,3852,3190
negative,436,552,480
neutral,2392,3160,2646


## Hypothesis Testing

In [97]:
import numpy as np
import pandas as pd

# Contingency table
# data = np.array([
#     [370, 242, 409],  # Positive
#     [53, 645, 209],   # Negative
#     [139, 419, 389]   # Neutral
# ])

data_mvsam = orientation_df.copy()

# Optional: Set labels
sentiments = ['positive', 'negative', 'neutral']
orientations = ['vertical', 'horizontal', 'box']

# df = pd.DataFrame(data, index=sentiments, columns=orientations)
# Contingency table

# Chi-square test
chi2, p, dof, expected = chi2_contingency(data_mvsam)

print(f"Chi-square statistic = {chi2:.3f}")
print(f"p-value = {p}")
print(f"Degrees of freedom = {dof}")

Chi-square statistic = 0.780
p-value = 0.941160398497051
Degrees of freedom = 4


In [98]:
print("Expected frequencies:")
pd.DataFrame(np.round(expected, 2), index=sentiments, columns=orientations)

Expected frequencies:


,vertical,horizontal,box
positive,2897.15,3833.14,3200.70
negative,428.26,566.61,473.13
neutral,2391.59,3164.24,2642.17


In [99]:
# p-value
p

0.941160398497051

In [100]:
# Decision
alpha = 0.05
if p < alpha:
    print("Reject the null hypothesis: There is a relationship between orientation and sentiment.")
else:
    print("Fail to reject the null hypothesis: No significant relationship found.")

Fail to reject the null hypothesis: No significant relationship found.
